# Satellite Orbit Explorer · built during Space Career Week 2026 · GNSS Academy

We will build this educational orbit analysis notebook in 8 stages:
1. **Setup** (Imports, Constants, Config)
2. **Load GP JSON** (Retrieve General Perturbations raw elements data)
3. **Parse Elements** (Extract Keplerian elements from GP data)
4. **Time Window** (Configure propagation interval and time steps)
5. **Propagate** (Run the two-body orbital model)
6. **Orbit Summary** (Analyze key parameters like apogee, perigee, and period)
7. **Plots** (Visualize ground track and 3D trajectory)
8. **KML and Report** (Export results to Google Earth and print final summary)

In [ ]:
import json
import math
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import requests

# Educational constants for a two-body Kepler orbit with spherical Earth
MU_EARTH_KM3_S2 = 398600.4418
EARTH_RADIUS_KM = 6378.137

# Configure Directories
BASE_DIR = Path('/content/satellite_orbits')
IN_DIR = BASE_DIR / 'input'
OUT_DIR = BASE_DIR / 'output'

IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Propagation configuration
STEP_SECONDS = 5
DURATION_HOURS = 24
LIVE_GROUP = None

# Print configuration summary
print("=== Stage 1 Setup Complete ===")
print(f"Input directory:  {IN_DIR}")
print(f"Output directory: {OUT_DIR}")
print(f"Propagation Step: {STEP_SECONDS} seconds")
print(f"Duration:         {DURATION_HOURS} hours")
print(f"Live Group Filter:{LIVE_GROUP}")

=== Stage 1 Setup Complete ===
Input directory:  /content/satellite_orbits/input
Output directory: /content/satellite_orbits/output
Propagation Step: 5 seconds
Duration:         24 hours
Live Group Filter:None


In [ ]:
# =====================================================================
# Stage 2: get the input files and load the GP records
# =====================================================================
INPUT_FILES = ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']
UPLOAD_DIRS = [Path('/content')]              # where the Colab Files panel puts dragged files
GP_FIELDS = ['OBJECT_NAME', 'NORAD_CAT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
             'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY']


def find_input_zip():
    """Return the first complete orbit_workshop_inputs*.zip in UPLOAD_DIRS, or None."""
    for folder in UPLOAD_DIRS:
        for path in sorted(folder.glob('orbit_workshop_inputs*.zip')):
            if zipfile.is_zipfile(path):        # False while the file is still uploading
                return path
    return None


def extract_inputs(timeout_s=900):
    """Wait for the input ZIP, then copy its three known files into IN_DIR (skipped if already there)."""
    if all((IN_DIR / name).exists() for name in INPUT_FILES):
        return
    print("Drag orbit_workshop_inputs.zip into the /content folder of the Files panel (folder icon, left).")
    print("This cell continues by itself as soon as the ZIP has arrived...")
    zip_path, waited = find_input_zip(), 0
    while zip_path is None:
        if waited >= timeout_s:
            raise FileNotFoundError("orbit_workshop_inputs.zip not found after 15 minutes: add it and run this cell again.")
        time.sleep(2)
        waited += 2
        zip_path = find_input_zip()
    with zipfile.ZipFile(zip_path) as zf:
        for name in INPUT_FILES:                # extract only the files we know
            (IN_DIR / name).write_bytes(zf.read(name))
    print(f"Extracted {len(INPUT_FILES)} files from {zip_path.name}.")


def read_gp_json(path):
    """Load a GP JSON file (a list of OMM records) and check that every record has the fields we use."""
    records = json.loads(Path(path).read_text(encoding='utf-8'))
    if not isinstance(records, list) or not records:
        raise ValueError(f"{path} must contain a non-empty JSON list of GP records.")
    for rec in records:
        missing = [k for k in GP_FIELDS if k not in rec]
        if missing:
            raise ValueError(f"{rec.get('OBJECT_NAME', '?')} is missing the fields {missing}.")
    return records


def print_sats(sats):
    for s in sats:
        print(f"  {s['OBJECT_NAME']:<24} NORAD {s['NORAD_CAT_ID']:<7} EPOCH {s['EPOCH']}")


extract_inputs()
sats = read_gp_json(IN_DIR / 'orbital_elements.json')
data_manifest = json.loads((IN_DIR / 'data_manifest.json').read_text(encoding='utf-8'))
source_note = f"Workshop snapshot: native CelesTrak GP JSON retrieved {data_manifest['retrieved_utc']}"
print(source_note)
print_sats(sats)


Drag orbit_workshop_inputs.zip into the /content folder of the Files panel (folder icon, left).
This cell continues by itself as soon as the ZIP has arrived...


FileNotFoundError: orbit_workshop_inputs.zip not found after 15 minutes: add it and run this cell again.

In [ ]:
from pathlib import Path

print("Checking Stage 2 files:")

for name in ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']:
    path = IN_DIR / name
    print(name, "FOUND" if path.exists() else "MISSING")

In [ ]:
from pathlib import Path

for folder in Path('/content').rglob('orbital_elements.json'):
    print("Found:", folder)


In [ ]:
from pathlib import Path

for name in ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']:
    matches = list(Path('/content').rglob(name))
    print(f"\n{name}:")
    for path in matches:
        print(path)

In [ ]:
from pathlib import Path

for path in Path('/content').rglob('data_manifest.json'):
    print("FOUND HERE:", path)

In [ ]:
from pathlib import Path
import shutil

source = Path('/data_manifest.json')
destination = IN_DIR / 'data_manifest.json'

shutil.copy2(source, destination)

print("Copied successfully:", destination.exists())

In [ ]:
for name in ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']:
    path = IN_DIR / name
    print(name, "FOUND" if path.exists() else "MISSING")

In [ ]:
import shutil
from pathlib import Path

shutil.copy2('/content/orbital_elements.json', IN_DIR / 'orbital_elements.json')
shutil.copy2('/content/world_map.txt', IN_DIR / 'world_map.txt')

print("Both files copied successfully!")

In [ ]:
for name in ['orbital_elements.json', 'data_manifest.json', 'world_map.txt']:
    print(name, "FOUND" if (IN_DIR / name).exists() else "MISSING")

### Stage 3: Parsing Keplerian Elements from GP Data

Here is a mapping of the OMM GP fields to their standard orbital symbols and the conversions needed for our physical model:

| GP Field | Symbol | Description | Input Unit | Model Unit | Conversion |
| :--- | :---: | :--- | :---: | :---: | :--- |
| `MEAN_MOTION` | $n$ | Mean angular velocity | revs/day | rad/s | $\times 2\pi / 86400$ |
| `ECCENTRICITY` | $e$ | Orbit eccentricity | decimal | decimal | None |
| `INCLINATION` | $i$ | Orbital plane tilt | degrees | radians | $\times \pi / 180$ |
| `RA_OF_ASC_NODE` | $\Omega$ | Right Ascension of Ascending Node | degrees | radians | $\times \pi / 180$ |
| `ARG_OF_PERICENTER` | $\omega$ | Argument of Perigee | degrees | radians | $\times \pi / 180$ |
| `MEAN_ANOMALY` | $M_0$ | Mean Anomaly at epoch | degrees | radians | $\times \pi / 180$ |
| `EPOCH` | $t_0$ | Reference date/time | ISO 8601 | Datetime | UTC parsed |

Using Kepler's Third Law, we compute the semi-major axis ($a$) in kilometers as:
$$a = \left( \frac{\mu}{n^2} \right)^{1/3}$$

> **Note on Accuracy**: General Perturbations (GP) datasets represent "mean" orbital elements generated specifically for the SGP4 mathematical model (which includes Earth's oblateness ($J_2$, $J_4$) and atmospheric drag). Plugging these mean elements directly into a simplified, unperturbed two-body Keplerian propagator provides an educational approximation of the orbit, rather than high-precision operational tracking.

### Why OMM JSON is better than the old TLE format
Orbit Mean-Elements Message (OMM) in JSON format is far easier to parse than the traditional TLE because it provides clearly named, self-documenting key-value pairs instead of rigid column-based plain text. Additionally, it natively supports standardized ISO 8601 date strings for EPOCH and allows explicit representation of six-digit satellite catalog numbers without truncation or custom encoding.

In [ ]:
def parse_gp_epoch(text):
    """Parses ISO 8601 UTC epoch strings into UTC-aware datetime objects."""
    # Remove trailing Z if present and force UTC timezone
    clean_text = text.rstrip('Z')
    return datetime.fromisoformat(clean_text).replace(tzinfo=timezone.utc)

def parse_gp_elements(rec):
    """Converts a single GP/OMM JSON record into Keplerian parameters in model units."""
    name = rec['OBJECT_NAME']
    norad_id = int(rec['NORAD_CAT_ID'])
    epoch = parse_gp_epoch(rec['EPOCH'])

    # Convert mean motion from revolutions/day to radians/second
    mean_motion_rev_day = float(rec['MEAN_MOTION'])
    n_rad_s = (mean_motion_rev_day * 2 * math.pi) / 86400.0

    # Calculate Semi-Major Axis (a) using Kepler's Third Law: a = (mu / n^2)**(1/3)
    a_km = (MU_EARTH_KM3_S2 / (n_rad_s ** 2)) ** (1/3)

    # Eccentricity is already decimal
    e = float(rec['ECCENTRICITY'])

    # Convert degrees to radians
    i_rad = math.radians(float(rec['INCLINATION']))
    raan_rad = math.radians(float(rec['RA_OF_ASC_NODE']))
    argp_rad = math.radians(float(rec['ARG_OF_PERICENTER']))
    M0_rad = math.radians(float(rec['MEAN_ANOMALY']))

    return {
        'name': name,
        'norad_id': norad_id,
        'label': f"{name} [{norad_id}]",
        'epoch': epoch,
        'n_rad_s': n_rad_s,
        'a_km': a_km,
        'e': e,
        'i_rad': i_rad,
        'raan_rad': raan_rad,
        'argp_rad': argp_rad,
        'M0_rad': M0_rad
    }

def elements_table(elements_list):
    """Returns lines of a formatted text table describing parsed Keplerian elements."""
    lines = []
    header = f"{'Satellite Name':<18} | {'NORAD':<6} | {'Epoch (UTC)':<19} | {'a (km)':<9} | {'e':<7} | {'i (deg)':<7} | {'RAAN':<6} | {'ARG':<6} | {'M0':<6}"
    divider = "-" * len(header)
    lines.append(header)
    lines.append(divider)

    for el in elements_list:
        row = (f"{el['name']:<18} | "
               f"{el['norad_id']:<6} | "
               f"{el['epoch'].strftime('%Y-%m-%d %H:%M:%S'):<19} | "
               f"{el['a_km']:<9.2f} | "
               f"{el['e']:<7.5f} | "
               f"{math.degrees(el['i_rad']):<7.2f} | "
               f"{math.degrees(el['raan_rad']):<6.1f} | "
               f"{math.degrees(el['argp_rad']):<6.1f} | "
               f"{math.degrees(el['M0_rad']):<6.1f}")
        lines.append(row)
    return lines

# Ensure sats list exists from Stage 2
if 'sats' not in globals() or not sats:
    raise NameError("Missing GP satellite data from Stage 2. Please run prior stages first.")

# Parse Keplerian elements
elements = [parse_gp_elements(s) for s in sats]

# Display the structured elements table
for line in elements_table(elements):
    print(line)

### Stage 4: Defining a Common Time Window and Evaluating Epoch Offsets

To compare and plot the satellite trajectories together, we establish a single, unified UTC time grid. However, because each satellite's orbital elements (GP/OMM record) are captured at a slightly different time (`epoch`), we must calculate the time offset between each satellite's element epoch and our simulation's start time.

This offset is critical because in a simple two-body Kepler model (and even in high-fidelity models), propagation error accumulates the further we propagate away from the measurement epoch. For instance, the workshop dataset includes a historical record for **QZS-2** dating back to **17 October 2026**; validating and highlighting stale elements (e.g., offsets greater than 7 days) alerts us to potential degradation in prediction accuracy.

In [ ]:
# Ensure elements are loaded from Stage 3
if 'elements' not in globals() or not elements:
    raise NameError("Missing Keplerian elements from Stage 3. Please run prior stages first.")

# 1. Determine simulation start: newest epoch in elements, floored to UTC midnight
newest_epoch = max(el['epoch'] for el in elements)
start_time_grid = newest_epoch.replace(hour=0, minute=0, second=0, microsecond=0)

# 2. Determine simulation end
end_time_grid = start_time_grid + timedelta(hours=DURATION_HOURS)

# 3. Build the common UTC time grid
total_seconds = int((end_time_grid - start_time_grid).total_seconds())
n_points = (total_seconds // STEP_SECONDS) + 1

times = [start_time_grid + timedelta(seconds=i * STEP_SECONDS) for i in range(n_points)]

# Print time window metrics
print("=== Stage 4 Time Grid Configuration ===")
print(f"Newest Satellite Epoch:    {newest_epoch}")
print(f"Simulation Start (Floored): {start_time_grid}")
print(f"Simulation End:             {end_time_grid}")
print(f"Total Data Points:          {n_points:,} (Step: {STEP_SECONDS}s, Duration: {DURATION_HOURS}h)\n")

# Evaluate and print time offsets for each satellite
print(f"{'Satellite Name':<18} | {'Epoch (UTC)':<19} | {'Offset (days)':<13} | {'Status'}")
print("-" * 65)
for el in elements:
    offset_delta = start_time_grid - el['epoch']
    offset_days = offset_delta.total_seconds() / 86400.0

    # Mark records older than 7 days as stale
    status = "STALE RECORD!" if abs(offset_days) > 7.0 else "Fresh"

    print(f"{el['name']:<18} | {el['epoch'].strftime('%Y-%m-%d %H:%M:%S'):<19} | {offset_days:<13.3f} | {status}")
print("-" * 65)

### Stage 5: Keplerian Orbit Propagation and Coordinate Rotation

In this stage, we implement the core physics of our orbital simulation to find each satellite's Earth-fixed position (latitude, longitude, and altitude) over our time grid:

1. **Solve Kepler's Equation**: We find the Eccentric Anomaly ($E$) from the Mean Anomaly ($M$) using Newton-Raphson iteration on $M = E - e\sin E$.
2. **Determine Position in Orbit**: We compute the distance ($r$), the True Anomaly ($\nu$), and the argument of latitude ($u = \omega + \nu$) in the orbital plane.
3. **Transform to Inertial Space**: We rotate the orbital plane into Earth-Centered Inertial (ECI) coordinates using standard rotation matrices with the Inclination ($i$) and Right Ascension of the Ascending Node ($\Omega$).
4. **Rotate to Earth-Fixed Frame (ECEF)**: To determine where the satellite is relative to observers on Earth, we compute the Greenwich Mean Sidereal Time (GMST) to rotate the Earth-fixed frame relative to the inertial frame. This accounts for Earth's rotation.
5. **Spherical Geocentric Projection**: Finally, we convert these rotating Cartesian coordinates to Latitude, Longitude, Altitude, and calculate the orbital speed using the Vis-Viva equation ($v = \sqrt{\mu(2/r - 1/a)}$).

In [ ]:
def solve_kepler(M, e):
    """Solves Kepler's Equation M = E - e*sin(E) using Newton-Raphson method."""
    # Initial guess recommendation
    E = math.pi if e > 0.8 else M
    tol = 1e-12
    max_iter = 50
    for _ in range(max_iter):
        f = E - e * math.sin(E) - M
        df = 1.0 - e * math.cos(E)
        delta = f / df
        E -= delta
        if abs(delta) < tol:
            return E
    raise RuntimeError(f"Kepler solver failed to converge: M={M}, e={e}")

def gmst_rad(t):
    """Calculates Greenwich Mean Sidereal Time (GMST) in radians for a given UTC datetime."""
    # J2000 epoch is 2000-01-01 12:00:00 UTC
    j2000 = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
    d = (t - j2000).total_seconds() / 86400.0
    T = d / 36525.0
    # IAU-1982 formulation
    theta_deg = 280.46061837 + 360.98564736629 * d + 0.000387933 * (T ** 2) - (T ** 3) / 38710000.0
    return math.radians(theta_deg % 360.0)

def propagate(el, time_grid):
    """Propagates satellite orbital elements over a grid of UTC datetimes."""
    trajectory = []
    a = el['a_km']
    e = el['e']
    i = el['i_rad']
    raan = el['raan_rad']
    argp = el['argp_rad']
    M0 = el['M0_rad']
    n = el['n_rad_s']
    epoch = el['epoch']

    for t in time_grid:
        dt_sec = (t - epoch).total_seconds()

        # 1. Mean Anomaly
        M = (M0 + n * dt_sec) % (2 * math.pi)

        # 2. Eccentric Anomaly
        E = solve_kepler(M, e)

        # 3. Position and true anomaly in orbital plane
        r = a * (1.0 - e * math.cos(E))
        sin_nu = (math.sqrt(1.0 - e**2) * math.sin(E))
        cos_nu = (math.cos(E) - e)
        nu = math.atan2(sin_nu, cos_nu)
        u = argp + nu

        # 4. Geocentric Inertial Position (ECI)
        x_eci = r * (math.cos(raan) * math.cos(u) - math.sin(raan) * math.sin(u) * math.cos(i))
        y_eci = r * (math.sin(raan) * math.cos(u) + math.cos(raan) * math.sin(u) * math.cos(i))
        z_eci = r * math.sin(u) * math.sin(i)

        # 5. Rotate to Earth-Fixed Frame (ECEF) by Greenwich Mean Sidereal Time
        theta = gmst_rad(t)
        x_ecef = math.cos(theta) * x_eci + math.sin(theta) * y_eci
        y_ecef = -math.sin(theta) * x_eci + math.cos(theta) * y_eci
        z_ecef = z_eci

        # 6. Spherical Earth Coordinates
        lat = math.atan2(z_ecef, math.hypot(x_ecef, y_ecef))
        lon = math.atan2(y_ecef, x_ecef)
        alt = r - EARTH_RADIUS_KM

        # 7. Vis-Viva speed
        speed = math.sqrt(MU_EARTH_KM3_S2 * (2.0 / r - 1.0 / a))

        trajectory.append({
            'time': t,
            'lat_deg': math.degrees(lat),
            'lon_deg': math.degrees(lon),
            'alt_km': alt,
            'speed_km_s': speed
        })
    return trajectory

# --- Validation and Assertions ---
# Verify GMST at J2000 is approximately 280.46 degrees
j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
gmst_j2000_deg = math.degrees(gmst_rad(j2000_epoch)) % 360.0
assert abs(gmst_j2000_deg - 280.46061837) < 1e-4, f"GMST test failed: {gmst_j2000_deg}"

# Verify Kepler Solver Residuals
for test_M in [0.1, 1.5, 3.0]:
    for test_e in [0.01, 0.1, 0.5]:
        test_E = solve_kepler(test_M, test_e)
        residual = abs(test_E - test_e * math.sin(test_E) - test_M)
        assert residual < 1e-10, f"Kepler solver accuracy test failed. Residual: {residual}"

# Ensure times and elements exist from previous steps
if 'times' not in globals() or 'elements' not in globals():
    raise NameError("Missing simulation grid or satellite elements. Run previous stages first.")

# Propagate tracks for all loaded satellites
tracks = [propagate(el, times) for el in elements]

# Print tracking checkpoint results for validation
print("=== Stage 5 Propagation Checkpoints ===")
for el, track in zip(elements, tracks):
    print(f"\nSatellite: {el['label']}")
    print("-" * 80)
    for index, label in [(0, 'START '), (len(times)//2, 'MIDDLE'), (len(times)-1, 'END   ')]:
        pt = track[index]
        print(f"{label} -> Time: {pt['time']} | Lat: {pt['lat_deg']:>7.2f}° | Lon: {pt['lon_deg']:>7.2f}° | Alt: {pt['alt_km']:>8.1f} km | Speed: {pt['speed_km_s']:.3f} km/s")
    print("-" * 80)

# Validate geostationary longitude if SES-5 is in the set
ses5_track = [track for el, track in zip(elements, tracks) if el['norad_id'] == 38652]
if ses5_track:
    for pt in ses5_track[0]:
        lon = pt['lon_deg']
        assert 4.0 <= lon <= 6.0, f"SES-5 Geostationary drift check failed: Longitude={lon:.3f}° E (expected ~5.0° E)"
    print("\nSuccess: SES-5 geostationary longitude constraint verified (stays between 4.0° and 6.0° East).")

### Stage 6: Orbit Summary Analysis and Interpretation

By summarizing these orbital parameters, we can identify and classify different types of orbits:

* **PULSAR-0**: With an orbital period of approximately **1.58 hours** and an inclination around **97.5°**, this represents a low Earth orbit (LEO). The inclination $>90^\circ$ makes it a **retrograde, near-polar, sun-synchronous** orbit, ideal for global imaging.
* **Galileo-29**: With an orbital period of about **14.08 hours**, this is in a Medium Earth Orbit (MEO) characteristic of global navigation satellite systems (GNSS).
* **SES-5 & QZS-2**: Both have orbital periods of approximately **23.93 hours** (one sidereal day), indicating geosynchronous orbits.
  * **SES-5** has near-zero eccentricity and inclination, making it a classic **Geostationary (GEO)** satellite hovering over a fixed point on the equator.
  * **QZS-2** has a significant eccentricity ($e \approx 0.075$) and inclination ($i \approx 39^\circ$). This is an **Inclined Geosynchronous Orbit (IGSO)** designed to spend more time at high elevations over specific regions (like Japan) in a figure-8 ground track.

In [ ]:
def orbit_summary(el, pts):
    """Calculates critical orbital parameters and speeds for a satellite orbit."""
    a = el['a_km']
    e = el['e']
    n = el['n_rad_s']

    # Calculate orbital period in hours
    period_h = (2.0 * math.pi / n) / 3600.0

    # Calculate perigee and apogee altitudes above spherical Earth
    perigee_km = a * (1.0 - e) - EARTH_RADIUS_KM
    apogee_km = a * (1.0 + e) - EARTH_RADIUS_KM

    # Extract minimum and maximum speeds from the propagated points
    speeds = [pt['speed_km_s'] for pt in pts]
    min_speed = min(speeds)
    max_speed = max(speeds)

    return {
        'name': el['name'],
        'norad_id': el['norad_id'],
        'period_h': period_h,
        'perigee_km': perigee_km,
        'apogee_km': apogee_km,
        'e': e,
        'i_deg': math.degrees(el['i_rad']),
        'min_speed_km_s': min_speed,
        'max_speed_km_s': max_speed
    }

def summary_table(summaries_list):
    """Formats and returns a text table summarizing the orbital profiles."""
    lines = []
    header = f"{'Satellite Name':<18} | {'NORAD':<6} | {'Period (h)':<10} | {'Perigee (km)':<12} | {'Apogee (km)':<11} | {'e':<7} | {'i (deg)':<7} | {'Vmin (km/s)':<11} | {'Vmax (km/s)':<11}"
    divider = "-" * len(header)
    lines.append(header)
    lines.append(divider)

    for s in summaries_list:
        row = (f"{s['name']:<18} | "
               f"{s['norad_id']:<6} | "
               f"{s['period_h']:<10.2f} | "
               f"{s['perigee_km']:<12.1f} | "
               f"{s['apogee_km']:<11.1f} | "
               f"{s['e']:<7.5f} | "
               f"{s['i_deg']:<7.2f} | "
               f"{s['min_speed_km_s']:<11.3f} | "
               f"{s['max_speed_km_s']:<11.3f}")
        lines.append(row)
    return lines

# Ensure dependencies are loaded from prior stages
if 'elements' not in globals() or 'tracks' not in globals():
    raise NameError("Missing elements or propagated tracks. Please run preceding stages.")

# Compute summaries for each satellite
summaries = [orbit_summary(el, pts) for el, pts in zip(elements, tracks)]

# Display the table
print("=== Stage 6 Orbit Summaries ===")
for line in summary_table(summaries):
    print(line)

# Assert period constraint for Galileo 29 (NORAD 59598) if present in the loaded dataset
galileo_sum = [s for s in summaries if s['norad_id'] == 59598]
if galileo_sum:
    g_period = galileo_sum[0]['period_h']
    assert 14.0 <= g_period <= 14.2, f"Assertion Failed: Galileo 29 period is {g_period:.3f} h (expected 14.0 - 14.2 h)"
    print(f"\nSuccess: Galileo 29 period of {g_period:.2f} h validated within specs!")

### Stage 7: Orbit Visualization & Interpretation

In this stage, we construct two scientific plots to visualize the ground tracks and velocity profiles of our satellites. Here is what we expect to observe on the ground tracks:

* **PULSAR-0**: A Low Earth Orbit (LEO) satellite that circles the Earth quickly. Because the Earth rotates underneath it as it completes multiple orbits, each subsequent pass shifts progressively westward.
* **Galileo-29**: An MEO satellite that traces out a smooth, wide wave pattern spanning the globe over its ~14-hour period.
* **SES-5**: A Geostationary (GEO) satellite that appears as a single static point near $5^\circ$ East on the equator because its orbital period perfectly matches Earth's rotation rate.
* **QZS-2**: An Inclined Geosynchronous Orbit (IGSO). It has a 24-hour period, a $39^\circ$ inclination, and an eccentric orbit. This combination creates an asymmetric "figure-8" ground track. It moves slower and spends more time at its apogee in the Northern Hemisphere (providing focused coverage over East Asia).

#### Why do eccentric orbits speed up near perigee?
According to **Kepler's Second Law** (conservation of angular momentum), a satellite must sweep out equal areas in equal times. When the satellite is closer to Earth (at *perigee*), the gravitational pull is stronger, requiring a higher velocity to maintain the orbit. As it travels to its furthest point (*apogee*), gravitational potential energy increases while kinetic energy decreases, causing the satellite to slow down.

In [ ]:
COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

def load_coastline(path):
    """Loads world_map.txt containing 'latitude longitude' pairs separated by blank lines."""
    segments = []
    current_seg_lons = []
    current_seg_lats = []

    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_seg_lons:
                    segments.append((current_seg_lons, current_seg_lats))
                    current_seg_lons = []
                    current_seg_lats = []
            else:
                # Format is: latitude longitude
                lat_str, lon_str = line.split()
                current_seg_lats.append(float(lat_str))
                current_seg_lons.append(float(lon_str))

    if current_seg_lons:
        segments.append((current_seg_lons, current_seg_lats))

    return segments

# Ensure prerequisites are defined
world_map_path = IN_DIR / 'world_map.txt'
if not world_map_path.is_file():
    raise FileNotFoundError("world_map.txt is missing from input directory!")

coastlines = load_coastline(world_map_path)

# --- Figure 1: Ground Tracks Plot ---
plt.figure(figsize=(14, 8))

# Draw coastlines in gray
for lons, lats in coastlines:
    plt.plot(lons, lats, color='gray', linewidth=0.8, alpha=0.7)

# Plot each satellite's ground track using scatter to avoid cross-meridian wrap lines
for idx, (el, track) in enumerate(zip(elements, tracks)):
    color = COLORS[idx % len(COLORS)]
    lons = [pt['lon_deg'] for pt in track]
    lats = [pt['lat_deg'] for pt in track]
    plt.scatter(lons, lats, color=color, s=2, label=el['label'])

plt.title("Satellite Ground Tracks (24-Hour Propagation)", fontsize=14, pad=15)
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(markerscale=6, loc='lower left')
plt.tight_layout()

ground_track_fig_path = OUT_DIR / 'ground_tracks.png'
plt.savefig(ground_track_fig_path, dpi=200)
plt.show()
print(f"Saved Ground Track plot to: {ground_track_fig_path}")

# --- Figure 2: Velocity Timeseries Plot ---
fig, axs = plt.subplots(len(elements), 1, figsize=(12, 10), sharex=True)
if len(elements) == 1:
    axs = [axs]

for idx, (el, track) in enumerate(zip(elements, tracks)):
    color = COLORS[idx % len(COLORS)]
    plot_times = [pt['time'] for pt in track]
    speeds = [pt['speed_km_s'] for pt in track]

    ax = axs[idx]
    ax.plot(plot_times, speeds, color=color, linewidth=2, label=el['name'])
    ax.set_ylabel("Speed\n(km/s)", fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right')
    # Prevent scientific offset notation on the vertical axis
    ax.get_yaxis().get_major_formatter().set_useOffset(False)

axs[-1].set_xlabel("Time (UTC)", fontsize=11)
plt.suptitle("Satellite Orbital Velocity Profiles", fontsize=14, y=0.98)
plt.tight_layout()

velocity_fig_path = OUT_DIR / 'velocity_timeseries.png'
plt.savefig(velocity_fig_path, dpi=200)
plt.show()
print(f"Saved Velocity Timeseries plot to: {velocity_fig_path}")

In [ ]:
import xml.etree.ElementTree as ET

def hex_to_kml_color(hex_str):
    """Converts a standard CSS hex string (#RRGGBB) to KML aabbggrr format."""
    hex_str = hex_str.lstrip('#')
    if len(hex_str) == 6:
        r, g, b = hex_str[0:2], hex_str[2:4], hex_str[4:6]
        return f"ff{b}{g}{r}"
    return "ffffffff"

# --- 1. Write orbit_tracks_ecef.kml ---
ET.register_namespace('', "http://www.opengis.net/kml/2.2")
kml_3d = ET.Element('kml')
doc_3d = ET.SubElement(kml_3d, 'Document')
name_3d = ET.SubElement(doc_3d, 'name')
name_3d.text = "Multi-Satellite Orbits | Earth-fixed"

for idx, (el, track) in enumerate(zip(elements, tracks)):
    color_hex = mcolors.to_hex(COLORS[idx % len(COLORS)])
    kml_color = hex_to_kml_color(color_hex)
    style_id = f"style_{el['norad_id']}"

    # Shared Style definition
    style = ET.SubElement(doc_3d, 'Style', id=style_id)

    line_style = ET.SubElement(style, 'LineStyle')
    line_color = ET.SubElement(line_style, 'color')
    line_color.text = kml_color
    line_width = ET.SubElement(line_style, 'width')
    line_width.text = "3"

    icon_style = ET.SubElement(style, 'IconStyle')
    icon_color = ET.SubElement(icon_style, 'color')
    icon_color.text = kml_color
    icon_scale = ET.SubElement(icon_style, 'scale')
    icon_scale.text = "1.2"
    icon_node = ET.SubElement(icon_style, 'Icon')
    icon_href = ET.SubElement(icon_node, 'href')
    icon_href.text = "http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png"

    # Placemark for line
    pm = ET.SubElement(doc_3d, 'Placemark')
    pm_name = ET.SubElement(pm, 'name')
    pm_name.text = el['name']
    pm_style_url = ET.SubElement(pm, 'styleUrl')
    pm_style_url.text = f"#{style_id}"

    ls = ET.SubElement(pm, 'LineString')
    ls_ext = ET.SubElement(ls, 'extrude')
    ls_ext.text = "1"
    ls_mode = ET.SubElement(ls, 'altitudeMode')
    ls_mode.text = "absolute"

    coords_node = ET.SubElement(ls, 'coordinates')
    coord_strings = []
    for pt in track:
        # Altitude must be converted from km to METRES
        coord_strings.append(f"{pt['lon_deg']:.6f},{pt['lat_deg']:.6f},{pt['alt_km'] * 1000.0:.1f}")
    coords_node.text = " ".join(coord_strings)

kml_3d_tree = ET.ElementTree(kml_3d)
kml_3d_path = OUT_DIR / 'orbit_tracks_ecef.kml'
with open(kml_3d_path, 'wb') as f:
    kml_3d_tree.write(f, encoding='utf-8', xml_declaration=True)
print(f"Saved static 3D KML to: {kml_3d_path}")

# --- 2. Write orbit_animation.kml ---
ET.register_namespace('', "http://www.opengis.net/kml/2.2")
ET.register_namespace('gx', "http://www.google.com/kml/ext/2.2")

kml_anim = ET.Element('kml')
doc_anim = ET.SubElement(kml_anim, 'Document')
name_anim = ET.SubElement(doc_anim, 'name')
name_anim.text = "Multi-Satellite Ground Tracks | Animation"

for idx, (el, track) in enumerate(zip(elements, tracks)):
    color_hex = mcolors.to_hex(COLORS[idx % len(COLORS)])
    kml_color = hex_to_kml_color(color_hex)
    style_id = f"style_anim_{el['norad_id']}"

    style = ET.SubElement(doc_anim, 'Style', id=style_id)
    line_style = ET.SubElement(style, 'LineStyle')
    line_color = ET.SubElement(line_style, 'color')
    line_color.text = kml_color
    line_width = ET.SubElement(line_style, 'width')
    line_width.text = "3"

    icon_style = ET.SubElement(style, 'IconStyle')
    icon_color = ET.SubElement(icon_style, 'color')
    icon_color.text = kml_color
    icon_node = ET.SubElement(icon_style, 'Icon')
    icon_href = ET.SubElement(icon_node, 'href')
    icon_href.text = "http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png"

    pm = ET.SubElement(doc_anim, 'Placemark')
    pm_name = ET.SubElement(pm, 'name')
    pm_name.text = el['name']
    pm_style_url = ET.SubElement(pm, 'styleUrl')
    pm_style_url.text = f"#{style_id}"

    multi_track = ET.SubElement(pm, '{http://www.google.com/kml/ext/2.2}MultiTrack')
    mt_interpolate = ET.SubElement(multi_track, '{http://www.google.com/kml/ext/2.2}interpolate')
    mt_interpolate.text = "1"

    # Downsample track: Keep one point every 60s (using step size to step correctly)
    step_index_multiplier = max(1, 60 // STEP_SECONDS)
    downsampled = track[::step_index_multiplier]

    # Segment tracks whenever longitude jumps by > 180 degrees to prevent wrap-around lines across the map
    sub_tracks = []
    current_sub_track = []
    for i, pt in enumerate(downsampled):
        if current_sub_track:
            prev_lon = current_sub_track[-1]['lon_deg']
            if abs(pt['lon_deg'] - prev_lon) > 180.0:
                sub_tracks.append(current_sub_track)
                current_sub_track = []
        current_sub_track.append(pt)
    if current_sub_track:
        sub_tracks.append(current_sub_track)

    for segment in sub_tracks:
        gxtrack = ET.SubElement(multi_track, '{http://www.google.com/kml/ext/2.2}Track')
        t_mode = ET.SubElement(gxtrack, 'altitudeMode')
        t_mode.text = "clampToGround"

        for pt in segment:
            when_val = ET.SubElement(gxtrack, 'when')
            when_val.text = pt['time'].strftime('%Y-%m-%dT%H:%M:%SZ')

        for pt in segment:
            coord_val = ET.SubElement(gxtrack, '{http://www.google.com/kml/ext/2.2}coord')
            coord_val.text = f"{pt['lon_deg']:.6f} {pt['lat_deg']:.6f} 0"

kml_anim_tree = ET.ElementTree(kml_anim)
kml_anim_path = OUT_DIR / 'orbit_animation.kml'
with open(kml_anim_path, 'wb') as f:
    kml_anim_tree.write(f, encoding='utf-8', xml_declaration=True)
print(f"Saved animation KML to: {kml_anim_path}")

# --- 3. Write orbit_report.txt ---
report_path = OUT_DIR / 'orbit_report.txt'
with open(report_path, 'w') as r_file:
    r_file.write("Satellite Orbit Report - built during Space Career Week 2026, GNSS Academy\n")
    r_file.write("=" * 85 + "\n\n")
    r_file.write(f"{source_note}\n")
    r_file.write(f"Simulation Window: {start_time_grid} to {end_time_grid} UTC\n")
    r_file.write(f"Step size:         {STEP_SECONDS} seconds\n")
    r_file.write("Model Caveat:      This is an educational two-body Kepler approximation (spherical Earth, basic rotation, SGP4 mean-elements input).\n\n")

    r_file.write("=== Orbit Summary Profiles ===\n")
    for line in summary_table(summaries):
        r_file.write(line + "\n")
    r_file.write("\n")

    r_file.write("=== Reference Orbital Elements (at Epoch) ===\n")
    for line in elements_table(elements):
        r_file.write(line + "\n")

print(f"Saved text report to: {report_path}")

# --- 4. Package Outputs as ZIP ---
zip_path = OUT_DIR / 'satellite_orbit_outputs.zip'
with zipfile.ZipFile(zip_path, 'w') as zip_f:
    zip_f.write(OUT_DIR / 'ground_tracks.png', arcname='ground_tracks.png')
    zip_f.write(OUT_DIR / 'velocity_timeseries.png', arcname='velocity_timeseries.png')
    zip_f.write(kml_3d_path, arcname='orbit_tracks_ecef.kml')
    zip_f.write(kml_anim_path, arcname='orbit_animation.kml')
    zip_f.write(report_path, arcname='orbit_report.txt')

print(f"\nZip archive created successfully at: {zip_path}")

In [ ]:
# Cell for secure file download triggers in Google Colab
from google.colab import files
try:
    files.download(str(zip_path))
except Exception as e:
    print(f"Download trigger failed (normal if not running interactively): {e}")

### Run-All Completion Checklist

- [x] **Stage 1**: Configured libraries, constants, and folder structures.
- [x] **Stage 2**: Imported raw OMM elements and validated the dataset schema.
- [x] **Stage 3**: Extracted standard Keplerian dimensions using Kepler's Third Law.
- [x] **Stage 4**: Generated a common, aligned UTC time grid.
- [x] **Stage 5**: Programmed the Newton-Raphson Kepler solver and Earth Greenwich Sidereal rotation.
- [x] **Stage 6**: Compiled a summary of orbital parameters (period, apses, limits).
- [x] **Stage 7**: Visualized 2D global ground tracks and physical velocity profiles.
- [x] **Stage 8**: Created static KML files, animations, reports, and bundled them for download.

### How to view your orbits in Google Earth Pro:
1. Download and extract the generated `satellite_orbit_outputs.zip` file on your local machine.
2. Open **Google Earth Pro** (Desktop App) or the web-based version.
3. Import **`orbit_tracks_ecef.kml`** via *File > Open* to view the static 3D orbital trajectories suspended in Earth-fixed space.
4. Import **`orbit_animation.kml`** to load the dynamic ground tracks. Use the **Time Slider** interface in the top-left corner of the Google Earth map viewer to play, pause, loop, or adjust the speed of the satellite ground tracking animation!